In [4]:
import os
from pathlib import Path

import pandas as pd

# Resolve the project data folder robustly from the notebook working directory
cwd = Path.cwd().resolve()
candidate_data_dirs = [
    cwd / "docs" / "data",
    cwd.parent / "docs" / "data",
    Path("..") / "docs" / "data",
]

data_folder = next((p.resolve() for p in candidate_data_dirs if p.exists()), None)
if data_folder is None:
    raise FileNotFoundError(
        f"Could not locate docs/data from working directory: {cwd}"
    )

career_path = data_folder / "career_master_11pt_vector_library.csv"
profiles_path = data_folder / "refined_riasec_student_profiles_normalized_v1.csv"

if not career_path.exists() or not profiles_path.exists():
    raise FileNotFoundError(
        f"Expected files not found in {data_folder}. "
        f"Missing: {[str(p) for p in [career_path, profiles_path] if not p.exists()]}"
    )

# 1. Load the updated datasets
career_library = pd.read_csv(career_path)
user_profiles = pd.read_csv(profiles_path)

# 2. Re-run the column check
riasec_cols = ["Realistic", "Investigative", "Artistic", "Social", "Enterprising", "Conventional"]
personality_cols = ["Extraversion", "Agreeableness", "Conscientiousness", "Emotional_Stability", "Openness"]
vector_cols = riasec_cols + personality_cols

print(f"✅ Files loaded successfully from: {data_folder}")

✅ Files loaded successfully from: C:\Users\grosh\GitHub Repositories\DSML 4360 Senior Exp\Find-My-Major-ML-Extension\docs\data


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def get_top_matches(user_vector, library_df, vector_cols, top_k=50):
    """
    Calculates similarity and returns the top K career matches.
    """
    # 1. Extract career vectors from library
    career_vectors = library_df[vector_cols].values
    
    # 2. Calculate Cosine Similarity
    # reshape(1, -1) converts the single user row into a 2D array for the math
    similarities = cosine_similarity(user_vector.reshape(1, -1), career_vectors)[0]
    
    # 3. Attach scores and sort
    results = library_df.copy()
    results['similarity_score'] = similarities
    
    return results.sort_values(by='similarity_score', ascending=False).head(top_k)

# --- Test the Matcher ---
# Take the very first user profile as a test case
test_user_vector = user_profiles[vector_cols].iloc[0].values

# Get the Top 50 (Caching for future re-ranking)
top_50_matches = get_top_matches(test_user_vector, career_library, vector_cols)

print(f"Top 3 Baseline Matches for Test User:")
print(top_50_matches[['Title', 'similarity_score']].head(3))

Top 3 Baseline Matches for Test User:
                                    Title  similarity_score
286       History Teachers, Postsecondary          0.917511
259  Architecture Teachers, Postsecondary          0.917320
271     Geography Teachers, Postsecondary          0.915767


In [7]:
# 1. Define a list of diverse user indices (based on your dataset)
# These indices are chosen to show different "Primary" interests
test_indices = [314, 42, 263, 79] 

# 2. Run the Loop to Check Results
for idx in test_indices:
    # Get user profile and their top interest
    user_row = user_profiles.iloc[idx]
    primary_interest = user_row[riasec_cols].idxmax()
    
    print(f"\n" + "="*50)
    print(f"SANITY CHECK: USER INDEX {idx} (Primary: {primary_interest})")
    print("="*50)
    
    # Display the User's Vector (6 RIASEC + 5 Personality)
    print("User Profile Scores:")
    display(user_row[vector_cols].to_frame().T)
    
    # Generate Top 10 Matches
    user_vec = user_row[vector_cols].values
    matches = get_top_matches(user_vec, career_library, vector_cols, top_k=10)
    
    print(f"\nTop 10 Career Matches for User {idx}:")
    print(matches[['Title', 'similarity_score']].to_string(index=False))


SANITY CHECK: USER INDEX 314 (Primary: Realistic)
User Profile Scores:


,Realistic,Investigative,Artistic,Social,Enterprising,Conventional,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,Openness
314,1.0,0.95,1.0,0.95,1.0,1.0,0.785714,0.714286,0.928571,0.928571,0.928571



Top 10 Career Matches for User 314:
                                 Title  similarity_score
                      Park Naturalists          0.974199
                  Landscape Architects          0.965711
Architects, Except Landscape and Naval          0.960615
                    Interior Designers          0.954260
   Commercial and Industrial Designers          0.949196
    Media Technical Directors/Managers          0.948996
           Urban and Regional Planners          0.946659
             Set and Exhibit Designers          0.942509
              Cooks, Private Household          0.941893
   Web and Digital Interface Designers          0.940891

SANITY CHECK: USER INDEX 42 (Primary: Artistic)
User Profile Scores:


,Realistic,Investigative,Artistic,Social,Enterprising,Conventional,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,Openness
42,0.825,0.725,1.0,0.95,0.85,0.975,0.857143,0.428571,0.928571,0.785714,0.928571



Top 10 Career Matches for User 42:
                                 Title  similarity_score
                      Park Naturalists          0.966778
Architects, Except Landscape and Naval          0.956434
                    Interior Designers          0.954344
                  Landscape Architects          0.954221
             Set and Exhibit Designers          0.952212
    Media Technical Directors/Managers          0.946120
   Commercial and Industrial Designers          0.944226
                              Curators          0.940350
                  Video Game Designers          0.938568
                         Art Directors          0.938405

SANITY CHECK: USER INDEX 263 (Primary: Social)
User Profile Scores:


,Realistic,Investigative,Artistic,Social,Enterprising,Conventional,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,Openness
263,0.325,0.975,0.375,1.0,0.375,0.775,0.785714,0.642857,0.928571,0.571429,0.857143



Top 10 Career Matches for User 263:
                                                                 Title  similarity_score
                          Mathematical Science Teachers, Postsecondary          0.993343
                                     Economics Teachers, Postsecondary          0.991347
                               Library Science Teachers, Postsecondary          0.990661
                              Computer Science Teachers, Postsecondary          0.989560
                                           Law Teachers, Postsecondary          0.986221
                                     Geography Teachers, Postsecondary          0.986183
                                       Physics Teachers, Postsecondary          0.985437
                                     Chemistry Teachers, Postsecondary          0.983706
                         Environmental Science Teachers, Postsecondary          0.983394
Atmospheric, Earth, Marine, and Space Sciences Teachers, Postsecondary   

,Realistic,Investigative,Artistic,Social,Enterprising,Conventional,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,Openness
79,0.425,1.0,0.35,0.8,0.275,0.525,0.642857,0.5,0.642857,0.357143,0.5



Top 10 Career Matches for User 79:
                                                                 Title  similarity_score
                          Mathematical Science Teachers, Postsecondary          0.976290
                                       Physics Teachers, Postsecondary          0.975464
                         Environmental Science Teachers, Postsecondary          0.975262
             Forestry and Conservation Science Teachers, Postsecondary          0.974474
                                     Chemistry Teachers, Postsecondary          0.974105
                            Biological Science Teachers, Postsecondary          0.972112
Atmospheric, Earth, Marine, and Space Sciences Teachers, Postsecondary          0.971683
                               Library Science Teachers, Postsecondary          0.971432
                                     Geography Teachers, Postsecondary          0.971370
                   Anthropology and Archeology Teachers, Postsecondary    

In [13]:
def re_rank_with_rejection(rejected_title, current_top_50, library_df, vector_cols, penalty_weight=0.15):
    # 1. Get the vector of the rejected career
    rejected_vector = library_df[library_df['Title'] == rejected_title][vector_cols].values[0]
    
    # 2. Filter out the REJECTED job so it can't come back
    remaining_top_50 = current_top_50[current_top_50['Title'] != rejected_title].copy()
    
    # 3. Calculate penalty based on similarity to the rejected job
    other_vectors = remaining_top_50[vector_cols].values
    sim_to_rejected = cosine_similarity(rejected_vector.reshape(1, -1), other_vectors)[0]
    
    # 4. Apply penalty to the original score
    remaining_top_50['adjusted_score'] = remaining_top_50['similarity_score'] - (sim_to_rejected * penalty_weight)
    
    return remaining_top_50.sort_values(by='adjusted_score', ascending=False)

# Let's say User 314 rejects the #1 result: 'Park Naturalists'
rejected_job = "Park Naturalists"
print(f"Action: User rejected '{rejected_job}'")

updated_list = re_rank_with_rejection(rejected_job, top_50_matches, career_library, vector_cols)

print("\nNew Top 3 after Re-Ranking:")
print(updated_list[['Title', 'adjusted_score']].head(10))

Action: User rejected 'Park Naturalists'

New Top 3 after Re-Ranking:
                                                 Title  adjusted_score
286                    History Teachers, Postsecondary        0.771665
259               Architecture Teachers, Postsecondary        0.770428
271                  Geography Teachers, Postsecondary        0.769228
282      Art, Drama, and Music Teachers, Postsecondary        0.766301
288  Family and Consumer Sciences Teachers, Postsec...        0.764649
268  Anthropology and Archeology Teachers, Postseco...        0.763111
269  Area, Ethnic, and Cultural Studies Teachers, P...        0.762712
289  Recreation and Fitness Studies Teachers, Posts...        0.762200
274                  Sociology Teachers, Postsecondary        0.761504
266      Environmental Science Teachers, Postsecondary        0.761095
